# Audio Inpainting with PlayDiffusion

Run Cell 1 (install) — wait for it to finish, then restart runtime when prompted
Run Cell 1 again (it will skip installs and just confirm everything is ready)
Run cells 4 through 10 sequentially

This notebook runs the full audio inpainting pipeline on Google Colab:
1. **Setup** — clone PlayDiffusion, install matching PyTorch 2.6.0 + fairseq2 + dependencies
2. **STT + LLM** — transcribe corrupted audio with Whisper, fix gaps with Gemini
3. **Inpainting** — use PlayDiffusion to fill in the corrupted audio regions

**Requirements:** Colab GPU runtime (Runtime → Change runtime type → T4 GPU)

In [ ]:
import os

# ── 1. Clone PlayDiffusion ──────────────────────────────────────────────
if not os.path.exists('/content/PlayDiffusion'):
    !git clone https://github.com/playht/PlayDiffusion.git
else:
    print("PlayDiffusion already cloned — skipping.")

# ── 2. Check whether dependencies are already installed (post-restart) ──
_need_install = True
try:
    import torch
    if torch.__version__.startswith('2.6.0'):
        import torchaudio, fairseq2
        _need_install = False
        print("Dependencies already installed. Skip to the next cell.")
except ImportError:
    pass

if _need_install:
    print("Installing dependencies — this may take a few minutes ...\n")

    # Remove Colab-default packages that conflict with PlayDiffusion
    !pip uninstall -y torch torchvision torchaudio fairseq2 fairseq2n 2>/dev/null

    # Step A: Install PyTorch 2.6.0 first
    !pip install torch==2.6.0 torchaudio==2.6.0 \
        --index-url https://download.pytorch.org/whl/cu124

    # Step B: Install fairseq2 explicitly using the PyTorch 2.6.0 + CUDA 12.4 index
    # Enforce torch==2.6.0 so fairseq2n doesn't pull a newer version
    !pip install fairseq2==0.4.4 fairseq2n==0.4.4 torch==2.6.0 torchaudio==2.6.0 \
        --extra-index-url https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cu124

    # Step C: Install PlayDiffusion runtime deps
    !pip install \
        "numpy>=1.26,<2.1" \
        "scipy>=1.12" \
        "scikit-learn>=1.3" \
        nltk jiwer pydantic soundfile boto3 tqdm python-decouple \
        safetensors tokenizers librosa einops huggingface-hub unidecode \
        torchtune==0.6.1 torchao==0.11.0 torch==2.6.0 torchaudio==2.6.0
    !pip install "syllables @ git+https://github.com/playht/python-syllables.git"

    # [NEW] Qwen3-ASR — SOTA open-source ASR with native word-level timestamps
    # Replaces Whisper for higher-quality transcription of corrupted audio.
    !pip install -q -U qwen-asr

    # [NEW] Ensure transformers is new enough for Qwen2-Audio-7B-Instruct
    !pip install -q -U transformers

    # Step D: Install Whisper (STT) + Gemini (LLM)
    !pip install openai-whisper whisper-timestamped google-generativeai torch==2.6.0 torchaudio==2.6.0

    print("\n" + "=" * 60)
    print("  RESTART THE RUNTIME NOW")
    print("  Runtime -> Restart runtime, then run the cells below.")
    print("=" * 60)

Cloning into 'PlayDiffusion'...
remote: Enumerating objects: 145, done.
remote: Counting objects: 100% (78/78), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 145 (delta 46), reused 37 (delta 37), pack-reused 67 (from 1)
Receiving objects: 100% (145/145), 75.23 KiB | 12.54 MiB/s, done.
Resolving deltas: 100% (53/53), done.
Installing dependencies — this may take a few minutes ...

Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128
Found existing installation: torchaudio 2.10.0+cu128
Uninstalling torchaudio-2.10.0+cu128:
  Successfully uninstalled torchaudio-2.10.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 112.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Looking in indexes: https://pypi.org/simple, https://fair.pkg.atmeta.com/fairseq2/whl/pt2.6.0/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.6/512.6 kB 35.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 128.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 138.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.2/179.2 kB 24.4 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0
    Uninstalling packaging-26.0:
      Successfully uninstalled packaging-26.0
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.9/14.9 MB 146.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 134.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 12.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
  Cloning https://github.com/playht/python-syllables.git to /tmp/pip-install-wbz62dbj/syllables_1c74ea10ce854bea885c852af059fcd8
  Running command git clone --filter=blob:none --quiet https://github.com/playht/python-syllables.git /tmp/pip-install-wbz62dbj/syllables_1c74ea10ce854bea885c852af059fcd8
  Resolved https://github.com/playht/python-syllables.git to com


  RESTART THE RUNTIME NOW
  Runtime -> Restart runtime, then run the cells below.


### After running the cell above for the first time:
1. **Restart the runtime** — Runtime -> Restart runtime
2. Then continue running from the cell below (you do **not** need to re-run the install cell)

### Step 2: Verify Installation

In [ ]:
import sys
sys.path.insert(0, '/content/PlayDiffusion/src')

import torch, torchaudio
print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"Torchaudio: {torchaudio.__version__}")
print(f"CUDA:       {torch.cuda.is_available()}  (toolkit {torch.version.cuda})")

import fairseq2
print(f"fairseq2:   {fairseq2.__version__}")

from playdiffusion import PlayDiffusion, InpaintInput
print("\nPlayDiffusion imported successfully!")

Python:     3.12.13
PyTorch:    2.6.0+cu124
Torchaudio: 2.6.0+cu124
CUDA:       True  (toolkit 12.4)
fairseq2:   0.4.4

PlayDiffusion imported successfully!


### Step 3: Prepare Data
Mount Google Drive and set the input/output directories. Upload your preprocessed audio files and JSON metadata to the input directory.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Point these at the folder where your preprocessed audio + JSON metadata live.
# Option A (Google Drive — persists between sessions):
input_dir  = "/content/drive/MyDrive/samples/preprocessed_audio"
output_dir = "/content/drive/MyDrive/samples/inpainted_audio"

os.makedirs(input_dir,  exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

print(f"Input dir:  {input_dir}")
print(f"Output dir: {output_dir}")

Mounted at /content/drive
Input dir:  /content/drive/MyDrive/samples/preprocessed_audio
Output dir: /content/drive/MyDrive/samples/inpainted_audio


### Step 4: Load Models & Helper Functions

This cell loads:
*   Qwen3-ASR-1.7B + ForcedAligner-0.6B — for speech-to-text with native word-level timestamps (replaces Whisper)
* Qwen3-32B via OpenRouter — for LLM transcript correction

[CHANGED] ASR switched from Whisper → Qwen3-ASR (SOTA open-source, better word boundaries, native forced alignment). Whisper is kept as fallback in case Qwen3-ASR fails to load (e.g. dependency / memory issue).

Helper functions defined here (used by Steps 5 and 6):
* transcribe_with_timestamps()  — unified interface (Qwen or Whisper)
* get_corrected_text_and_original_word_times()
* has_acoustic_discontinuity()
* smooth_silence_gap()


In [ ]:
import numpy as np
import librosa
import soundfile as sf
import requests
import torch
from google.colab import userdata

# ── [NEW] Load Qwen3-ASR (primary), fall back to Whisper if it fails ──────────
USE_QWEN_ASR = True          # Set False to force Whisper instead
QWEN_ASR_MODEL_ID = "Qwen/Qwen3-ASR-1.7B"      # Use "Qwen3-ASR-0.6B" if OOM
QWEN_ALIGNER_ID   = "Qwen/Qwen3-ForcedAligner-0.6B"

qwen_asr_model = None
whisper_model  = None

if USE_QWEN_ASR:
    try:
        print(f"Loading {QWEN_ASR_MODEL_ID} + forced aligner ...")
        from qwen_asr import Qwen3ASRModel
        qwen_asr_model = Qwen3ASRModel.from_pretrained(
            QWEN_ASR_MODEL_ID,
            dtype=torch.bfloat16,
            device_map="cuda:0",
            max_inference_batch_size=8,
            max_new_tokens=512,
            forced_aligner=QWEN_ALIGNER_ID,
            forced_aligner_kwargs=dict(
                dtype=torch.bfloat16,
                device_map="cuda:0",
            ),
        )
        print("Qwen3-ASR loaded.")
    except Exception as e:
        print(f"Qwen3-ASR failed to load ({e}) — falling back to Whisper.")
        qwen_asr_model = None

if qwen_asr_model is None:
    print("Loading Whisper model (fallback) ...")
    import whisper
    import whisper_timestamped as whisper_ts
    whisper_model = whisper.load_model("base")

# ── StackAI client — GPT-5.4 transcript correction ───────────────────────────
# Model to use for LLM correction.  Switch to "gpt_5_4_pro" for the Pro tier.
STACKAI_LLM_MODEL = "gpt_5_4"

_STACKAI_BASE = "https://api.stackai.com/inference/v0/run/{run_id}/{endpoint_id}"
_STACKAI_RUN_ID = "475c0540-6417-4995-ae46-62b1f9b8fe4a"

_MODEL_REGISTRY = {
    "gpt_5_4_pro": "69e7e0aef65bb6b52b53a555",
    "gpt_5_4":     "69e7e168bea566f9d2a4109b",
    "gpt_4o":      "69e7e1ab55e57fa84fc09ec2",
    "sonnet_4_6":  "69e7e2339505c1211767cbd4",
    "opus_4_6":    "69e7e326e31c3a23d8f8b33c",
    "gemini_3_5":  "69e7e35dbea566f9d2a4109c",
    "haiku_4_5":   "69e7e470a265535ccb488e95",
}


class StackAIClient:
    """Thin HTTP wrapper around the Stack AI inference bridge."""

    def __init__(self, token, run_id=_STACKAI_RUN_ID,
                 max_retries=3, retry_delay=5.0, timeout=120.0):
        import time as _time
        self._time = _time
        self._token = token
        self._run_id = run_id
        self._max_retries = max_retries
        self._retry_delay = retry_delay
        self._timeout = timeout
        self._headers = {
            "Authorization": f"Bearer {token}",
            "Content-Type": "application/json",
        }

    def query(self, model_name, prompt, user_id=""):
        """Send a single-turn prompt; return the response string."""
        if model_name not in _MODEL_REGISTRY:
            raise ValueError(f"Unknown model '{model_name}'. "
                             f"Available: {sorted(_MODEL_REGISTRY)}")
        endpoint_id = _MODEL_REGISTRY[model_name]
        url = _STACKAI_BASE.format(run_id=self._run_id, endpoint_id=endpoint_id)
        payload = {"user_id": user_id, "in-0": prompt}
        last_exc = None
        for attempt in range(1, self._max_retries + 1):
            try:
                resp = requests.post(url, headers=self._headers,
                                     json=payload, timeout=self._timeout)
                if 400 <= resp.status_code < 500:
                    resp.raise_for_status()
                if resp.status_code >= 500:
                    raise requests.HTTPError(f"HTTP {resp.status_code}", response=resp)
                return self._extract_text(resp.json())
            except (requests.Timeout, requests.ConnectionError, requests.HTTPError) as exc:
                last_exc = exc
                if attempt < self._max_retries:
                    print(f"  [stackai] attempt {attempt} failed ({exc}), "
                          f"retrying in {self._retry_delay}s...")
                    self._time.sleep(self._retry_delay)
        raise RuntimeError(f"Stack AI query failed after {self._max_retries} attempts: {last_exc}")

    @staticmethod
    def _extract_text(data):
        if isinstance(data.get("outputs"), dict):
            out = data["outputs"].get("out-0")
            if isinstance(out, str):
                return out
        if isinstance(data.get("out-0"), str):
            return data["out-0"]
        for key in ("response", "text", "content", "message", "result"):
            if isinstance(data.get(key), str):
                return data[key]
        raise ValueError(f"Cannot extract text from Stack AI response. "
                         f"Top-level keys: {list(data.keys())}")


_stackai_client = None
try:
    _stackai_client = StackAIClient(token=userdata.get('STACKAI_TOKEN'))
    print(f"StackAI configured — using {STACKAI_LLM_MODEL} (GPT-5.4) for correction.")
except Exception as e:
    print(f"Could not configure StackAI ({e}).")
    print("Running without LLM correction — raw ASR transcripts will be used.")


def call_llm(transcript):
    """Sends a broken transcript to GPT-5.4 via StackAI for correction."""
    if _stackai_client is None:
        return None
    prompt = (
        "The following text is a transcription of an audio file that had "
        "network connectivity drops. There are missing words or parts of words. "
        "Please guess what the full, logical sentence should be and return "
        "ONLY the complete, corrected sentence without any explanation.\n\n"
        f"Transcript: {transcript}"
    )
    try:
        return _stackai_client.query(STACKAI_LLM_MODEL, prompt)
    except Exception as e:
        print(f"  LLM error ({e}) — using raw transcript.")
        return None


# ── [NEW] Unified transcription function ──────────────────────────────────────
# Returns (transcript, word_timestamps) where word_timestamps is a list of
# {'word', 'start', 'end'} dicts — same format as before, regardless of
# whether Qwen3-ASR or Whisper was used underneath.
def transcribe_with_timestamps(audio_path):
    if qwen_asr_model is not None:
        # Qwen3-ASR path
        results = qwen_asr_model.transcribe(
            audio=audio_path,
            language="English",
            return_time_stamps=True,
        )
        r = results[0]
        transcript = r.text

        word_timestamps = []
        ts = getattr(r, 'time_stamps', None) or []
        # time_stamps can be a flat list of word objects or a list-of-lists;
        # handle both to be safe across package versions
        flat = ts[0] if (ts and isinstance(ts[0], (list, tuple))) else ts
        for w in flat:
            word = getattr(w, 'text', None) or (w.get('text') if isinstance(w, dict) else '')
            s    = getattr(w, 'start_time', None) or (w.get('start_time') if isinstance(w, dict) else 0.0)
            e    = getattr(w, 'end_time',   None) or (w.get('end_time')   if isinstance(w, dict) else 0.0)
            word_timestamps.append({'word': word.strip(), 'start': float(s), 'end': float(e)})
        return transcript, word_timestamps
    else:
        # Whisper fallback path
        audio     = whisper_ts.load_audio(audio_path)
        result_ts = whisper_ts.transcribe(whisper_model, audio, language="en", verbose=False)
        transcript = result_ts['text']
        word_timestamps = []
        for seg in result_ts.get('segments', []):
            for wi in seg.get('words', []):
                word_timestamps.append({
                    'word':  wi.get('text', '').strip(),
                    'start': wi['start'],
                    'end':   wi['end'],
                })
        return transcript, word_timestamps


def get_corrected_text_and_original_word_times(audio_path):
    """Transcribes audio, then corrects the transcript with the LLM."""
    print(f"  Transcribing ...")
    transcript, word_timestamps = transcribe_with_timestamps(audio_path)
    print(f"  Original:  {transcript}")

    corrected = call_llm(transcript)
    if corrected:
        print(f"  Corrected: {corrected}")
        return corrected, word_timestamps
    return transcript, word_timestamps


# ── Helper: MFCC discontinuity check at a known cut point ─────────────────────
# [IMPROVED] Adaptive threshold — no longer hardcoded.
# Computes the distribution of ALL frame-to-frame MFCC distances in the file,
# then flags the cut point only if its local jump exceeds the file's own
# percentile baseline. This adapts to speaker voice, codec, and channel noise.
def has_acoustic_discontinuity(audio_path, cut_time_sec,
                                n_mfcc=13, hop_sec=0.01, percentile=95.0):
    """Flag if the MFCC jump at cut_time_sec exceeds the file's 95th-pct baseline."""
    y, sr = librosa.load(audio_path, sr=None, mono=True)
    hop_length = int(hop_sec * sr)
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc, hop_length=hop_length)

    # Global baseline: distribution of all frame-to-frame L2 distances
    all_dists = np.linalg.norm(np.diff(mfccs, axis=1), axis=0)
    if len(all_dists) == 0:
        return False
    threshold = float(np.percentile(all_dists, percentile))

    # Local distance at the cut
    cut_frame = int(round(cut_time_sec / hop_sec))
    cut_frame = max(1, min(cut_frame, mfccs.shape[1] - 1))
    local_dist = float(np.linalg.norm(mfccs[:, cut_frame] - mfccs[:, cut_frame - 1]))

    return local_dist > threshold


# ── Helper: smooth a silence gap with a short crossfade ───────────────────────
def smooth_silence_gap(audio_path, start_sec, end_sec, output_path, crossfade_ms=20):
    y, sr = sf.read(audio_path)
    mono  = (y.ndim == 1)
    if mono:
        y = y[:, np.newaxis]
    s_sample = int(start_sec * sr)
    e_sample = int(end_sec   * sr)
    cf = min(int(crossfade_ms * sr / 1000), s_sample, max(0, len(y) - e_sample))
    if cf > 0:
        fade_out = np.linspace(1.0, 0.0, cf)[:, np.newaxis]
        fade_in  = np.linspace(0.0, 1.0, cf)[:, np.newaxis]
        blended  = y[s_sample - cf : s_sample] * fade_out + y[e_sample : e_sample + cf] * fade_in
        result   = np.concatenate([y[: s_sample - cf], blended, y[e_sample + cf :]])
    else:
        result = np.concatenate([y[:s_sample], y[e_sample:]])
    if mono:
        result = result[:, 0]
    sf.write(output_path, result, sr)

Loading Qwen/Qwen3-ASR-1.7B + forced aligner ...
Qwen3-ASR failed to load (check_model_inputs() missing 1 required positional argument: 'func') — falling back to Whisper.
Loading Whisper model (fallback) ...
Importing the dtw module. When using in academic works please cite:
  T. Giorgino. Computing and Visualizing Dynamic Time Warping Alignments in R: The dtw Package.
  J. Stat. Soft., doi:10.18637/jss.v031.i07.



100%|████████████████████████████████████████| 139M/139M [00:00<00:00, 442MiB/s]


Qwen (OpenRouter) configured.


### Step 5: Corruption Detection Module

This step adds automatic detection of corrupted audio regions. Previously, the pipeline required a JSON file with known timestamps. Now, if no events are found in the JSON, this module scans the audio automatically and produces the same (start_sec, end_sec) format.

Two complementary signal-processing methods are used:

**Method A — RMS Energy Dropout Detection**

Computes rolling loudness (dB) and flags sudden drops. Best at: full dropouts, hard silence insertions.

**Method B — MFCC Discontinuity Scan**

Computes frame-level acoustic feature distance and flags abrupt jumps.Best at: mid-word glitches, cut-and-splice artifacts.

Both methods run independently. Their results are merged into a unified list of (start_sec, end_sec) corruption candidates with padding applied.

______________________________________________________________________________

*Parameters can be tuned below if you get too many false positives (raise thresholds) or miss real glitches (lower thresholds).*

In [ ]:
import numpy as np
import librosa

# Tunable parameters

# Method A — RMS dropout detection
RMS_WINDOW_SEC      = 0.020   # Analysis window size (20ms)
RMS_HOP_SEC         = 0.010   # Hop between windows (10ms)
RMS_DROP_THRESHOLD  = 20.0    # dB drop vs local max that flags a dropout
RMS_MIN_DURATION    = 0.020   # Ignore detections shorter than this (seconds)

# Method B — MFCC discontinuity scan
MFCC_FRAME_SEC      = 0.025   # MFCC analysis window (25ms)
MFCC_HOP_SEC        = 0.010   # MFCC hop (10ms)
MFCC_N_COEFFS       = 13      # Number of MFCC coefficients
MFCC_DIFF_THRESHOLD = 30.0     # Fallback constant (used only if adaptive fails)
# [IMPROVED] The actual per-file threshold is computed adaptively inside
#            detect_mfcc_discontinuities() using the 97th-percentile of
#            this file's own frame-to-frame distances, overriding this value.
MAX_REGION_SEC      = 1.0    # [NEW] reject merged regions longer than this

# Merging and padding
MERGE_GAP_SEC       = 0.15    # Fuse detections closer than this (seconds)
CONTEXT_PAD_SEC     = 0.05    # Expand each detected region by this on each side


def detect_rms_dropouts(y, sr):
    """
    [Method A] Scans the audio for sudden drops in RMS energy.
    Returns list of (start_sec, end_sec) tuples.
    """
    win = int(RMS_WINDOW_SEC * sr)
    hop = int(RMS_HOP_SEC    * sr)
    eps = 1e-10

    # Compute RMS energy per frame in dB
    frames = librosa.util.frame(y, frame_length=win, hop_length=hop)
    rms    = np.sqrt(np.mean(frames ** 2, axis=0))
    rms_db = 20 * np.log10(rms + eps)
    times  = librosa.frames_to_time(np.arange(len(rms_db)), sr=sr, hop_length=hop)

    # Rolling max over ~0.3s window serves as the local loudness reference
    ref_frames  = max(1, int(0.3 / RMS_HOP_SEC))
    rolling_max = np.array([
        np.max(rms_db[max(0, i - ref_frames) : i + 1])
        for i in range(len(rms_db))
    ])

    # Flag any frame whose energy is significantly below the local max
    dropout_mask = (rolling_max - rms_db) > RMS_DROP_THRESHOLD

    # Convert frame-level mask to time intervals
    events     = []
    in_dropout = False
    start_idx  = 0
    for i, flag in enumerate(dropout_mask):
        if flag and not in_dropout:
            in_dropout = True
            start_idx  = i
        elif not flag and in_dropout:
            in_dropout = False
            if (times[i] - times[start_idx]) >= RMS_MIN_DURATION:
                events.append((times[start_idx], times[i]))
    if in_dropout and (times[-1] - times[start_idx]) >= RMS_MIN_DURATION:
        events.append((times[start_idx], times[-1]))

    return events


def detect_mfcc_discontinuities(y, sr):
    """
    [Method B] Scans the audio for abrupt frame-to-frame acoustic jumps.
    Returns list of (start_sec, end_sec) tuples centred on each detected jump.
    """
    hop   = int(MFCC_HOP_SEC   * sr)
    n_fft = int(MFCC_FRAME_SEC * sr)

    # Compute MFCCs and frame-to-frame L2 distance
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=MFCC_N_COEFFS,
                                  n_fft=n_fft, hop_length=hop)
    diffs = np.linalg.norm(np.diff(mfccs, axis=1), axis=0)
    times = librosa.frames_to_time(np.arange(len(diffs)), sr=sr, hop_length=hop)

    # [IMPROVED] Adaptive threshold: 97th percentile of this file's distances.
    # Replaces the global MFCC_DIFF_THRESHOLD constant — which is
    # speaker/codec-dependent and poorly calibrated on new audio.
    nonzero = diffs[diffs > 0]
    adaptive_threshold = (float(np.percentile(nonzero, 97.0))
                          if len(nonzero) > 0 else MFCC_DIFF_THRESHOLD)

    events = []
    for i, d in enumerate(diffs):
        if d > adaptive_threshold:
            t = float(times[i])
            events.append((
                max(0.0, t - CONTEXT_PAD_SEC),
                t + CONTEXT_PAD_SEC
            ))

    return events

def merge_events(events, audio_duration=None):
    """
    Merges overlapping or nearby (start, end) tuples and applies padding.
    Nearby means closer than MERGE_GAP_SEC.
    """
    if not events:
        return []

    events = sorted(events)
    merged = [list(events[0])]

    for start, end in events[1:]:
        if start - merged[-1][1] <= MERGE_GAP_SEC:
            merged[-1][1] = max(merged[-1][1], end)
        else:
            merged.append([start, end])

    # Apply context padding and clamp to audio duration
    result = []
    for start, end in merged:
        s = max(0.0, start - CONTEXT_PAD_SEC)
        e = end + CONTEXT_PAD_SEC
        if audio_duration:
            e = min(audio_duration, e)
        result.append((s, e))

    result = [(s, e) for (s, e) in result if (e - s) <= MAX_REGION_SEC]

    return result

def detect_corrupted_regions(audio_path, verbose=True):
    """
    Main entry point for automatic corruption detection.

    Runs Method A (RMS) and Method B (MFCC) on the audio, merges all
    candidates, and returns a list of dicts in the same format as the
    JSON events list:
        [{"start_sec": float, "end_sec": float}, ...]

    This allows Step 6 to use auto-detected events as a drop-in replacement
    for JSON-provided timestamps when no JSON is available.
    """
    y, sr    = librosa.load(audio_path, sr=None, mono=True)
    duration = len(y) / sr

    rms_events  = detect_rms_dropouts(y, sr)
    mfcc_events = detect_mfcc_discontinuities(y, sr)

    if verbose:
        print(f"  [Method A — RMS]  {len(rms_events)} candidate(s) found")
        print(f"  [Method B — MFCC] {len(mfcc_events)} candidate(s) found")

    merged = merge_events(rms_events + mfcc_events, audio_duration=duration)

    if verbose:
        print(f"  [Merged]          {len(merged)} region(s) after combining:")
        for s, e in merged:
            print(f"                    {s:.3f}s – {e:.3f}s")

    return [{"start_sec": s, "end_sec": e} for s, e in merged]

print("Step 5 ready: Corruption detection module loaded.")
print(f"  RMS drop threshold:  {RMS_DROP_THRESHOLD} dB")
print(f"  MFCC diff threshold: {MFCC_DIFF_THRESHOLD} L2 distance")
print(f"  Merge gap:           {MERGE_GAP_SEC}s")

Step 5 ready: Corruption detection module loaded.
  RMS drop threshold:  20.0 dB
  MFCC diff threshold: 30.0 L2 distance
  Merge gap:           0.15s


### Step 5b: Qwen2-Audio Detection (Optional, A100 recommended)

[NEW] Uses Qwen2-Audio-7B-Instruct — a multimodal LLM that accepts audio as input — to *reason* about the recording and identify dropout regions.

Unlike RMS+MFCC (pure signal processing) or Qwen3-ASR (pure transcription) this model simultaneously understands the ACOUSTIC content AND the LINGUISTIC context, so it can detect the "so__ wheat" case where ASR confidently misreads a mid-word glitch as a clean word.

Memory:    Qwen2-Audio-7B in bfloat16 uses ~14 GB of GPU RAM.
Recommended runtime: A100 (40 GB) with High-RAM enabled.
If it fails to load, set USE_QWEN_AUDIO = False below to skip it.

Output format: same as detect_corrupted_regions() — a list of {"start_sec", "end_sec"} dicts — so it plugs into the same evaluation machinery in Step 7.

In [ ]:
import json as _json
import re
import torch
import librosa

USE_QWEN_AUDIO = True   # Set False to skip Qwen2-Audio entirely

qwen_audio_model     = None
qwen_audio_processor = None

if USE_QWEN_AUDIO:
    try:
        print("Loading Qwen2-Audio-7B-Instruct (this may take a minute) ...")
        from transformers import Qwen2AudioForConditionalGeneration, AutoProcessor

        qwen_audio_processor = AutoProcessor.from_pretrained(
            "Qwen/Qwen2-Audio-7B-Instruct"
        )
        qwen_audio_model = Qwen2AudioForConditionalGeneration.from_pretrained(
            "Qwen/Qwen2-Audio-7B-Instruct",
            dtype=torch.bfloat16,
            device_map="cuda:0",
        )
        print("Qwen2-Audio loaded.")
    except Exception as e:
        print(f"Qwen2-Audio failed to load ({e}) — skipping this method.")
        qwen_audio_model = None


# ── Prompt template ───────────────────────────────────────────────────────────
# We ask for strict JSON so the output is machine-parseable. We include the
# audio duration so the model has a reference frame for its timestamps.
QWEN_AUDIO_PROMPT_TEMPLATE = """You are analyzing a speech recording that may contain brief network dropouts.
A dropout is a short region (typically 30-500 milliseconds) where the audio is
lost, cut off, glitches, or becomes unintelligible due to network issues.

The audio is {duration:.2f} seconds long.

Listen carefully and identify every dropout you can hear. For each one,
report its approximate start time and end time in seconds.

Respond with ONLY a valid JSON object in exactly this format, no other text:
{{"dropouts": [{{"start_sec": 1.23, "end_sec": 1.45}}]}}

If you don't hear any dropouts, respond with: {{"dropouts": []}}"""


def _parse_qwen_audio_response(response_text, audio_duration):
    """
    Parses the model's text response into a list of {start_sec, end_sec} dicts.
    Robust to common deviations (code fences, extra prose, alt key names).
    """
    # Strip code fences if present
    text = response_text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    # First try: full response as JSON
    try:
        data = _json.loads(text)
    except _json.JSONDecodeError:
        # Second try: extract the first {...} block
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            return []
        try:
            data = _json.loads(match.group(0))
        except _json.JSONDecodeError:
            return []

    raw = data.get("dropouts", []) if isinstance(data, dict) else []
    events = []
    for item in raw:
        if not isinstance(item, dict):
            continue
        s = item.get("start_sec", item.get("start"))
        e = item.get("end_sec",   item.get("end"))
        if s is None or e is None:
            continue
        try:
            s = max(0.0, float(s))
            e = min(audio_duration, float(e))
        except (TypeError, ValueError):
            continue
        if e > s:
            events.append({"start_sec": s, "end_sec": e})

    return events


def detect_with_qwen_audio(audio_path, verbose=True, debug=False):
    """
    Runs Qwen2-Audio-7B-Instruct on the audio file and returns a list of
    {"start_sec", "end_sec"} dicts for any dropouts it detects.

    Returns an empty list if Qwen2-Audio isn't loaded or if parsing fails.
    Set debug=True to print the model's raw response (useful for prompt tuning).
    """
    if qwen_audio_model is None:
        return []

    # Load audio at the processor's expected sample rate
    sr_target = qwen_audio_processor.feature_extractor.sampling_rate
    audio, sr = librosa.load(audio_path, sr=sr_target, mono=True)
    duration  = len(audio) / sr

    # Build the conversation
    prompt_text = QWEN_AUDIO_PROMPT_TEMPLATE.format(duration=duration)
    conversation = [
        {"role": "user", "content": [
            {"type": "audio", "audio_url": audio_path},
            {"type": "text",  "text": prompt_text},
        ]},
    ]
    text = qwen_audio_processor.apply_chat_template(
        conversation, add_generation_prompt=True, tokenize=False
    )

    inputs = qwen_audio_processor(
        text=text, audios=[audio], return_tensors="pt", padding=True,
        sampling_rate=sr_target,
    )
    inputs = {k: v.to(qwen_audio_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        generate_ids = qwen_audio_model.generate(**inputs, max_length=512)
    # Trim the prompt tokens from the output
    generate_ids = generate_ids[:, inputs["input_ids"].size(1):]
    response = qwen_audio_processor.batch_decode(
        generate_ids, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )[0]

    if debug:
        print(f"  [Qwen2-Audio raw response]: {response}")

    events = _parse_qwen_audio_response(response, duration)

    if verbose:
        print(f"  [Qwen2-Audio]    {len(events)} region(s) detected:")
        for e in events:
            print(f"                   {e['start_sec']:.3f}s – {e['end_sec']:.3f}s")

    return events


print("Step 5b ready.")
print(f"  Qwen2-Audio available: {qwen_audio_model is not None}")

Loading Qwen2-Audio-7B-Instruct (this may take a minute) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/853 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/876 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

Qwen2-Audio loaded.
Step 5b ready.
  Qwen2-Audio available: True


### Step 6: Evaluation Helpers

[NEW] When JSON events are available, we treat them as GROUND TRUTH and run the auto-detection (Step 5) in parallel to measure how well detection is doing on real data.

Metrics computed per file and aggregated at the end of Step 7:
* Recall    = fraction of JSON events that were detected
* Precision = fraction of detected events that match a real JSON event
* Mean IoU  = average Intersection-over-Union for matched pairs

An event is "matched" if its time range overlaps with a JSON event with IoU >= MATCH_IOU_THRESHOLD.

The results of Step 7 still use the JSON events for actual inpainting (so the repair is as correct as possible); detection runs purely for measurement.


In [ ]:
import numpy as np

# Lowered from 0.5 — dropouts are short, so even small timing offsets can
# push a valid match below 0.5 IoU. 0.3 is more forgiving while still
# enforcing meaningful overlap.
MATCH_IOU_THRESHOLD = 0.3


def _iou(a, b):
    """Intersection-over-Union of two (start, end) intervals."""
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = max(a[1], b[1]) - min(a[0], b[0])
    return inter / union if union > 0 else 0.0


def evaluate_detection(gt_events, detected_events):
    """
    Compares ground-truth JSON events to auto-detected events.
    Returns a dict with counts and recall / precision / mean_iou.

    Matching is greedy: for each GT event, pick the unmatched detection
    with the highest IoU. If that IoU >= MATCH_IOU_THRESHOLD, count as matched.
    """
    gt  = [(e["start_sec"], e["end_sec"]) for e in gt_events]
    det = [(e["start_sec"], e["end_sec"]) for e in detected_events]

    matched_gt  = set()
    matched_det = set()
    iou_values  = []

    for i, g in enumerate(gt):
        best_j, best_iou = -1, 0.0
        for j, d in enumerate(det):
            if j in matched_det:
                continue
            iou = _iou(g, d)
            if iou > best_iou:
                best_iou, best_j = iou, j
        if best_iou >= MATCH_IOU_THRESHOLD:
            matched_gt.add(i)
            matched_det.add(best_j)
            iou_values.append(best_iou)

    recall    = len(matched_gt)  / len(gt)  if gt  else 1.0
    precision = len(matched_det) / len(det) if det else 1.0
    mean_iou  = float(np.mean(iou_values)) if iou_values else 0.0

    return {
        "n_gt":       len(gt),
        "n_detected": len(det),
        "n_matched":  len(matched_gt),
        "recall":     recall,
        "precision":  precision,
        "mean_iou":   mean_iou,
    }


def print_aggregate_metrics(totals):
    """Pretty-print the running totals collected across all processed files."""
    if totals["n_gt"] == 0:
        print("\nNo ground-truth events available — skipping aggregate metrics.")
        return

    overall_recall    = totals["n_matched"]  / totals["n_gt"]
    overall_precision = (totals["n_matched"] / totals["n_detected"]
                          if totals["n_detected"] else 0.0)
    overall_mean_iou  = (totals["iou_sum"] / totals["iou_count"]
                          if totals["iou_count"] else 0.0)

    print(f"\n{'=' * 60}")
    print("AGGREGATE DETECTION METRICS")
    print(f"{'=' * 60}")
    print(f"  Total GT events:       {totals['n_gt']}")
    print(f"  Total detected events: {totals['n_detected']}")
    print(f"  Total matched:         {totals['n_matched']}")
    print(f"  Overall recall:        {overall_recall:.2%}")
    print(f"  Overall precision:     {overall_precision:.2%}")
    print(f"  Mean IoU (matched):    {overall_mean_iou:.3f}")


print("Step 6 ready: evaluation helpers loaded.")
print(f"  IoU match threshold: {MATCH_IOU_THRESHOLD}")

Step 6 ready: evaluation helpers loaded.
  IoU match threshold: 0.3


### Step 7: Run the Full Inpainting Pipeline

Loads PlayDiffusion and processes every sample in input_dir.
For each audio file it:

 1. Transcribes with Qwen3-ASR (or Whisper fallback) and corrects with Qwen LLM — all via the helpers defined in Step 4
 2. Runs automatic corruption detection (Step 5)
 3. If JSON events exist: evaluates detection vs ground truth (Step 6)  and uses the JSON events for actual inpainting
 If no JSON events: uses the auto-detected events
 4. Iterates over the chosen events:
    * Re-transcribes the current (partially repaired) temp file
    * Snaps event boundaries to real word boundaries
    * Silence gap   → crossfade smoothing (no PlayDiffusion)
    * Acoustic glitch invisible to ASR → forces text diff
    * Normal case   → calls PlayDiffusion to regenerate the region
 5. Saves the repaired file as <name>_inpainted.wav
 6. After all files: prints aggregate detection metrics (if JSON was used)

[CHANGED from previous version]
* ASR switched from Whisper → Qwen3-ASR (via Step 4's helpers)
* Detection now runs on EVERY file, not only when JSON is missing
* Evaluation metrics printed per file + aggregate at the end

In [ ]:
# ── Step 7: Run the Full Inpainting Pipeline ─────────────────────────────────
#
# Loads PlayDiffusion and processes every sample in input_dir.
# For each audio file it:
#
#   1. Transcribes with Qwen3-ASR (or Whisper fallback) and corrects with Qwen
#      LLM — all via the helpers defined in Step 4
#   2. Runs automatic corruption detection (Step 5 + optional Step 5b)
#   3. If JSON events exist: evaluates BOTH detection methods vs ground truth
#      and uses the JSON events for actual inpainting
#      If no JSON events: uses the signal-processing detected events
#   4. Iterates over the chosen events:
#        a. Re-transcribes the current (partially repaired) temp file
#        b. Snaps event boundaries to real word boundaries
#        c. Silence gap   → crossfade smoothing (no PlayDiffusion)
#        d. Acoustic glitch invisible to ASR → forces text diff
#        e. Normal case   → calls PlayDiffusion to regenerate the region
#   5. Saves the repaired file as <name>_inpainted.wav
#   6. After all files: prints aggregate detection metrics for BOTH methods
#      (signal-processing vs Qwen2-Audio) side by side
#
# [CHANGED from previous version]
#   - [NEW] Also evaluates Qwen2-Audio detection in parallel when available
#   - [NEW] Aggregate summary prints both methods' metrics for comparison
# ─────────────────────────────────────────────────────────────────────────────

import sys, os, json, glob
sys.path.insert(0, '/content/PlayDiffusion/src')

import numpy as np
import soundfile as sf
import torchaudio
from playdiffusion import PlayDiffusion, InpaintInput
import scipy.io.wavfile as wav

# ── Load PlayDiffusion ────────────────────────────────────────────────────────
print("Loading PlayDiffusion model ...")
inpainter = PlayDiffusion()
print("Model loaded.\n")

# ── Padding duration added to both ends of the temp file ─────────────────────
# Prevents NaN errors when a corruption event is very close to the start
# or end of the audio (PlayDiffusion needs surrounding context on both sides).
PAD_SEC = 0.1

# ── Running totals for aggregate detection metrics ────────────────────────────
# Signal-processing method (RMS + MFCC)
eval_totals = {
    "n_gt":       0,
    "n_detected": 0,
    "n_matched":  0,
    "iou_sum":    0.0,
    "iou_count":  0,
}

# [NEW] Qwen2-Audio method (only populated if Qwen2-Audio was loaded in Step 5b)
eval_totals_qwen = {
    "n_gt":       0,
    "n_detected": 0,
    "n_matched":  0,
    "iou_sum":    0.0,
    "iou_count":  0,
}

# ── Main loop ─────────────────────────────────────────────────────────────────
json_files = sorted(glob.glob(os.path.join(input_dir, "*.json")))

if not json_files:
    print(f"No JSON metadata files found in {input_dir}")
    print("Upload your preprocessed audio files + JSON metadata first.")

for meta_file in json_files:
    with open(meta_file, 'r') as f:
        metadata = json.load(f)

    audio_path = os.path.join(input_dir, os.path.basename(metadata.get("output", "")))

    print(f"{'=' * 60}")
    print(f"Processing: {os.path.basename(audio_path)}")

    if not os.path.exists(audio_path):
        print("  Audio file not found — skipping.\n")
        continue

    # ── Step 7a: Transcribe and correct ───────────────────────────────────────
    corrected_text, original_word_timestamps = get_corrected_text_and_original_word_times(audio_path)

    # ── Step 7b: Run detection methods + evaluate vs ground truth ─────────────
    gt_events = metadata.get("events", [])

    print("  Running automatic corruption detection (RMS + MFCC) ...")
    detected_events = detect_corrupted_regions(audio_path, verbose=True)

    # [NEW] Also run Qwen2-Audio detection if it's loaded
    detected_events_qwen = []
    if qwen_audio_model is not None:
        print("  Running Qwen2-Audio detection ...")
        detected_events_qwen = detect_with_qwen_audio(audio_path, verbose=True, debug=False)

    if gt_events:
        # ── Signal-processing method metrics ─────────────────────────────────
        metrics = evaluate_detection(gt_events, detected_events)
        print(f"  [Signal-Processing] GT: {metrics['n_gt']}  "
              f"Detected: {metrics['n_detected']}  Matched: {metrics['n_matched']}")
        print(f"  [Signal-Processing] Recall: {metrics['recall']:.2%}  "
              f"Precision: {metrics['precision']:.2%}  "
              f"Mean IoU: {metrics['mean_iou']:.3f}")

        eval_totals["n_gt"]       += metrics["n_gt"]
        eval_totals["n_detected"] += metrics["n_detected"]
        eval_totals["n_matched"]  += metrics["n_matched"]
        eval_totals["iou_sum"]    += metrics["mean_iou"] * metrics["n_matched"]
        eval_totals["iou_count"]  += metrics["n_matched"]

        # ── [NEW] Qwen2-Audio method metrics ─────────────────────────────────
        if qwen_audio_model is not None:
            metrics_q = evaluate_detection(gt_events, detected_events_qwen)
            print(f"  [Qwen2-Audio]       GT: {metrics_q['n_gt']}  "
                  f"Detected: {metrics_q['n_detected']}  Matched: {metrics_q['n_matched']}")
            print(f"  [Qwen2-Audio]       Recall: {metrics_q['recall']:.2%}  "
                  f"Precision: {metrics_q['precision']:.2%}  "
                  f"Mean IoU: {metrics_q['mean_iou']:.3f}")

            eval_totals_qwen["n_gt"]       += metrics_q["n_gt"]
            eval_totals_qwen["n_detected"] += metrics_q["n_detected"]
            eval_totals_qwen["n_matched"]  += metrics_q["n_matched"]
            eval_totals_qwen["iou_sum"]    += metrics_q["mean_iou"] * metrics_q["n_matched"]
            eval_totals_qwen["iou_count"]  += metrics_q["n_matched"]

        # Use JSON events for the actual inpainting (they're ground truth)
        events = gt_events
        print(f"  Using {len(events)} JSON event(s) for inpainting.")
    else:
        # No ground truth — fall back to signal-processing detection for inpainting
        events = detected_events
        print(f"  No JSON events — using {len(events)} auto-detected region(s).")

    base      = os.path.splitext(os.path.basename(audio_path))[0]
    temp_path = os.path.join(output_dir, f"{base}_temp.wav")

    # ── Step 7c: Write temp file with silence padding on both ends ────────────
    # This gives PlayDiffusion audio context even for cuts at the very start
    # or very end of the file, preventing NaN errors in those edge cases.
    y, sr = sf.read(audio_path)
    pad   = np.zeros(int(PAD_SEC * sr), dtype=y.dtype)
    if y.ndim == 1:
        y_padded = np.concatenate([pad, y, pad])
    else:
        pad2d    = np.zeros((int(PAD_SEC * sr), y.shape[1]), dtype=y.dtype)
        y_padded = np.concatenate([pad2d, y, pad2d])
    sf.write(temp_path, y_padded, sr)

    # Shift all event timestamps forward to account for the leading pad
    events_padded = [
        {"start_sec": e["start_sec"] + PAD_SEC, "end_sec": e["end_sec"] + PAD_SEC}
        for e in events
    ]

    # ── Step 7d: Inpaint each event ───────────────────────────────────────────
    for i, event in enumerate(events_padded):
        original_start_sec = event.get("start_sec")
        original_end_sec   = event.get("end_sec")

        # Re-transcribe the current temp file to get fresh word timestamps.
        # Earlier repairs in this file may have shifted timings slightly.
        current_transcript, current_word_timestamps = transcribe_with_timestamps(temp_path)

        # ── Find words that genuinely overlap the cut region ──────────────────
        # Collect every word with any overlap with [original_start, original_end].
        # If none overlap, the cut fell entirely in a silence gap.
        overlapping = [
            (idx, w) for idx, w in enumerate(current_word_timestamps)
            if w['start'] < original_end_sec and w['end'] > original_start_sec
        ]

        if overlapping:
            ov_indices, ov_words = zip(*overlapping)
            adjusted_start_sec = min(w['start'] for w in ov_words)
            adjusted_end_sec   = max(w['end']   for w in ov_words)
            in_silence = False
        else:
            adjusted_start_sec = original_start_sec
            adjusted_end_sec   = original_end_sec
            in_silence = True

        print(f"  Event {i+1}: original cut  {original_start_sec:.3f}s – {original_end_sec:.3f}s")
        print(f"             adjusted       {adjusted_start_sec:.3f}s – {adjusted_end_sec:.3f}s")

        # ── Branch 0: very short gap — crossfade regardless of content ─────────
        # [NEW] For gaps ≤ 20 ms, invoking PlayDiffusion + TTS is wasteful and
        # degrades speaker identity (TTS voice ≠ original caller). A linear
        # crossfade is perceptually transparent at these durations.
        gap_ms = (original_end_sec - original_start_sec) * 1000.0
        if gap_ms < 20.0:
            print(f"  → Very short gap ({gap_ms:.1f} ms < 20 ms): crossfade, skip TTS.")
            smooth_silence_gap(temp_path, adjusted_start_sec, adjusted_end_sec, temp_path)
            continue

        # ── Branch A: silence gap — crossfade instead of PlayDiffusion ────────
        if in_silence:
            print("  → Silence gap: applying crossfade smoothing.")
            smooth_silence_gap(temp_path, adjusted_start_sec, adjusted_end_sec, temp_path)
            continue

        # ── Branch B: mid-word artifact invisible to text diff ────────────────
        # If the word was correctly transcribed despite the cut, input_text
        # and output_text are identical for that word → PlayDiffusion sees
        # no diff and leaves the acoustic glitch in place.
        # Solution: if MFCC discontinuity is present at the cut, drop the
        # overlapping words from input_text so PlayDiffusion is forced to
        # regenerate them.
        if has_acoustic_discontinuity(temp_path, original_start_sec):
            keep_indices   = set(range(len(current_word_timestamps))) - set(ov_indices)
            modified_input = ' '.join(
                current_word_timestamps[idx]['word'] for idx in sorted(keep_indices)
            )
            print(f"  → Acoustic artifact detected — forcing text diff.")
            print(f"    Modified input_text: {modified_input}")
            input_text_for_inpaint = modified_input
        else:
            input_text_for_inpaint = current_transcript

        # ── Branch C: normal inpainting via PlayDiffusion ─────────────────────
        try:
            request = InpaintInput(
                audio=temp_path,
                input_text=input_text_for_inpaint,
                output_text=corrected_text,
                start_time=adjusted_start_sec,
                end_time=adjusted_end_sec,
                input_word_times=current_word_timestamps,
            )
            result_audio = inpainter.inpaint(request)
            sample_rate, audio_data = result_audio
            wav.write(temp_path, sample_rate, audio_data)

        except Exception as e:
            print(f"  FAILED: {e}")

    # ── Step 7e: Trim padding and save final output ───────────────────────────
    y_final, sr_final = sf.read(temp_path)
    pad_samples = int(PAD_SEC * sr_final)
    if y_final.ndim == 1:
        y_final = y_final[pad_samples : len(y_final) - pad_samples]
    else:
        y_final = y_final[pad_samples : len(y_final) - pad_samples, :]

    out_file = os.path.join(output_dir, f"{base}_inpainted.wav")
    sf.write(out_file, y_final, sr_final)
    os.remove(temp_path)
    print(f"  Saved: {out_file}\n")

# ── Step 7f: Print aggregate detection metrics for both methods ───────────────
print(f"\n{'=' * 60}")
print("AGGREGATE DETECTION METRICS — Signal Processing (RMS + MFCC)")
print(f"{'=' * 60}")
print_aggregate_metrics(eval_totals)

if qwen_audio_model is not None:
    print(f"\n{'=' * 60}")
    print("AGGREGATE DETECTION METRICS — Qwen2-Audio-7B-Instruct")
    print(f"{'=' * 60}")
    print_aggregate_metrics(eval_totals_qwen)

Loading PlayDiffusion model ...


v090_g_01105000:   0%|          | 0.00/1.67G [00:00<?, ?B/s]

tokenizer-multi_bpe16384_merged_extended(…):   0%|          | 0.00/1.14M [00:00<?, ?B/s]

xlsr2_1b_v2_custom.pt:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

kmeans_10k.npy:   0%|          | 0.00/51.2M [00:00<?, ?B/s]

...

  Original:   some value which exact value is 2 to the power 7. Now, this is known as the steady
  LLM error ('choices') — using raw transcript.
  Running automatic corruption detection (RMS + MFCC) ...
  [Method A — RMS]  12 candidate(s) found
  [Method B — MFCC] 408 candidate(s) found
  [Merged]          0 region(s) after combining:
  Running Qwen2-Audio detection ...
  [Qwen2-Audio]    0 region(s) detected:
  [Signal-Processing] GT: 2  Detected: 0  Matched: 0
  [Signal-Processing] Recall: 0.00%  Precision: 100.00%  Mean IoU: 0.000
  [Qwen2-Audio]       GT: 2  Detected: 0  Matched: 0
  [Qwen2-Audio]       Recall: 0.00%  Precision: 100.00%  Mean IoU: 0.000
  Using 2 JSON event(s) for inpainting.


100%|██████████| 807/807 [00:00<00:00, 2665.29frames/s]
/usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py:266: UserWarning: n_fft=2048 is too large for input signal of length=800
  warnings.warn(


  Event 1: original cut  2.048s – 2.083s
             adjusted       1.700s – 2.460s
  → Acoustic artifact detected — forcing text diff.
    Modified input_text: some value which is 2 to the power 7. Now, this is known as the steady
Input: output_text=' some value which exact value is 2 to the power 7. Now, this is known as the steady' num_steps=30 init_temp=1.0 init_diversity=1.0 guidance=0.5 rescale=0.7 topk=25 audio_token_syllable_ratio=None audio='/content/drive/MyDrive/samples/inpainted_audio/sample_10_temp.wav' input_text='some value which is 2 to the power 7. Now, this is known as the steady' input_word_times=[{'word': 'some', 'start': np.float64(0.0), 'end': np.float64(0.4)}, {'word': 'value', 'start': np.float64(0.4), 'end': np.float64(0.82)}, {'word': 'which', 'start': np.float64(0.82), 'end': np.float64(1.7)}, {'word': 'exact', 'start': np.float64(1.7), 'end': np.float64(2.06)}, {'word': 'value', 'start': np.float64(2.06), 'end': np.float64(2.46)}, {'word': 'is', 'start': np

100%|██████████| 912/912 [00:00<00:00, 2048.11frames/s]


  Event 2: original cut  7.868s – 7.939s
             adjusted       7.680s – 8.060s
  → Acoustic artifact detected — forcing text diff.
    Modified input_text: some value which exact value is 2 to the power 7. Now this is known static.
Input: output_text=' some value which exact value is 2 to the power 7. Now, this is known as the steady' num_steps=30 init_temp=1.0 init_diversity=1.0 guidance=0.5 rescale=0.7 topk=25 audio_token_syllable_ratio=None audio='/content/drive/MyDrive/samples/inpainted_audio/sample_10_temp.wav' input_text='some value which exact value is 2 to the power 7. Now this is known static.' input_word_times=[{'word': 'some', 'start': np.float64(0.0), 'end': np.float64(0.38)}, {'word': 'value', 'start': np.float64(0.38), 'end': np.float64(0.84)}, {'word': 'which', 'start': np.float64(0.84), 'end': np.float64(1.7)}, {'word': 'exact', 'start': np.float64(1.7), 'end': np.float64(2.02)}, {'word': 'value', 'start': np.float64(2.02), 'end': np.float64(2.54)}, {'word': 'is',

In [ ]:
# This clears the output folder !
# import glob, os

# for f in glob.glob(os.path.join(output_dir, "*")):
#     os.remove(f)

# print("Output folder cleared.")

Output folder cleared.
